# Denoising of '/home/jupyter-vruiz/gdrive_TomogramDenoising/tomograms/empiar10311_stack_crop.mrc using CryoCARE "Even/Odd Registered"

* Inputs: `../even_odd/even.mrc` and `../even_odd/odd.mrc`, the pair of noisy vols.
* Outputs: `denoised/empiar10311_stack_crop.mrc` with the denoised vol, and `denoised.pdf` with a view of a tile of a central slice in Z.

In [ ]:
from pathlib import Path
from collections import namedtuple
import json

In [ ]:
Args = namedtuple("args", ["original", "even", "odd", "even_reg", "odd_reg", "denoised"])
args = Args("/home/jupyter-vruiz/gdrive_TomogramDenoising/tomograms/empiar10311_stack_crop.mrc",
            "../even_odd/even.mrc", "../even_odd/odd.mrc",
            "even_registered.mrc", "odd_registered.mrc", 
            "denoised/denoised.mrc")

In [ ]:
if Path(args.denoised).exists():
    raise Exception(f"{args.denoised} already exists ... exiting")

In [ ]:
if not Path(args.even).exists() or not Path(args.odd):
    %run split_even_odd.ipynb
    Args = namedtuple("args", ["original", "even", "odd", "even_reg", "odd_reg", "denoised"])
    args = Args("/home/jupyter-vruiz/gdrive_TomogramDenoising/tomograms/empiar10311_stack_crop.mrc",
                "../even_odd/even.mrc", "../even_odd/odd.mrc",
                "even_registered.mrc", "odd_registered.mrc", 
                "denoised/denoised.mrc")

In [ ]:
if not Path(args.even_reg).exists() or not Path(args.odd_reg):
    %run project_odd_even_and_viceversa.ipynb
    Args = namedtuple("args", ["original", "even", "odd", "even_reg", "odd_reg", "denoised"])
    args = Args("/home/jupyter-vruiz/gdrive_TomogramDenoising/tomograms/empiar10311_stack_crop.mrc",
                "../even_odd/even.mrc", "../even_odd/odd.mrc",
                "even_registered.mrc", "odd_registered.mrc", 
                "denoised/denoised.mrc")

## Configure cryoCARE

%%writefile train_data_config__odd_even_registered.json
{
    "even": ["even_registered.mrc", "even.mrc"],
    "odd": ["odd.mrc", "odd_registered.mrc"],
    "mask": [""],
    "patch_shape": [16, 16, 16],
    "num_slices": 800,
    "split": 0.9,
    "tilt_axis": "Y",
    "n_normalization_samples": 200,
    "path": "./odd_even_registered_data",
    "overwrite": "True"  
}

%%writefile train_data_config__odd_even_registered.json
{
    "even": ["even.mrc"],
    "odd": ["odd_registered.mrc"],
    "mask": [""],
    "patch_shape": [16, 16, 16],
    "num_slices": 800,
    "split": 0.9,
    "tilt_axis": "Y",
    "n_normalization_samples": 200,
    "path": "./odd_even_registered_data",
    "overwrite": "True"  
}

%%writefile train_data_config__odd_even_registered.json
{
    "even": ["even.mrc", "odd_registered.mrc"],
    "odd": ["even_registered.mrc", "odd.mrc"],
    "mask": [""],
    "patch_shape": [16, 16, 16],
    "num_slices": 800,
    "split": 0.9,
    "tilt_axis": "Y",
    "n_normalization_samples": 200,
    "path": "./odd_even_registered_data",
    "overwrite": "True"  
}

%%writefile train_data_config__odd_even_registered.json
{
    "even": ["even_registered.mrc", "even.mrc"],
    "odd": ["odd.mrc", "odd_registered.mrc"],
    "mask": [""],
    "patch_shape": [16, 16, 16],
    "num_slices": 800,
    "split": 0.9,
    "tilt_axis": "Y",
    "n_normalization_samples": 200,
    "path": "./odd_even_registered_data",
    "overwrite": "True"  
}

In [ ]:
_ = {
    "even": [args.even, args.even_reg],
    "odd": [args.odd_reg, args.odd],
    "mask": [""],
    "patch_shape": [16, 16, 16],
    "num_slices": 800,
    "split": 0.9,
    "tilt_axis": "Y",
    "n_normalization_samples": 200,
    "path": "./data",
    "overwrite": "True"
}

with open("train_data_config.json", 'w') as f:
    json.dump(_, f, indent=4)

In [ ]:
%%bash
#cd /nas/vruiz/cryoCARE/empiar10311
source ~/envs/cryoCARE/bin/activate
cryoCARE_extract_train_data.py --conf train_data_config.json

## Train

In [ ]:
%%writefile train_config.json
{
  "train_data": "./data",
  "epochs": 50,
  "steps_per_epoch": 200,
  "batch_size": 16,
  "unet_kern_size": 3,
  "unet_n_depth": 3,
  "unet_n_first": 16,
  "learning_rate": 0.0004,
  "model_name": "model",
  "path": "./",
  "gpu_id": [1]
}

In [ ]:
%%bash
#cd /nas/vruiz/cryoCARE/empiar10311
source ~/envs/cryoCARE/bin/activate
cryoCARE_train.py --conf train_config.json

## Infer

%%writefile predict_config__odd_even_registered.json
{
    "path": "./model.tar.gz",
    "even": ["empiar10311_stack_crop.mrc"], 
    "odd": ["empiar10311_stack_crop.mrc"],
    "n_tiles": [1,1,1],
    "output": "denoised",
    "overwrite": "True",
    "gpu_id": [1]
}

In [ ]:
_ = {
    "path": "./model.tar.gz",
    "even": [args.original], 
    "odd": [args.original],
    "n_tiles": [1,1,1],
    "output": "denoised",
    "overwrite": "True",
    "gpu_id": [1]
}

with open("predict_config.json", 'w') as f:
    json.dump(_, f, indent=4)

In [ ]:
%%bash
#cd /nas/vruiz/cryoCARE/empiar10311
#pwd
source ~/envs/cryoCARE/bin/activate
cryoCARE_predict.py --conf predict_config.json || true

In [ ]:
import mrcfile
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def read_MRC(file_path):
    return mrcfile.read(file_path)

In [ ]:
original_vol = read_MRC(args.original)

In [ ]:
denoised_vol = read_MRC(args.denoised)

In [ ]:
# Choose a slice index in the middle of the volume for a good comparison
slice_idx = original_vol.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(20, 20))

# Plot the original slice z
im1 = axes[0].imshow(original_vol[slice_idx, :, :].T, cmap='gray', origin='lower')
axes[0].set_title(f'Original Slice Z={slice_idx}')
axes[0].grid(False)

# Plot the original slice z+1
im2 = axes[1].imshow(denoised_vol[slice_idx, :, :].T, cmap='gray', origin='lower')
axes[1].set_title(f'N2N-Odd-Even-Registered Denoised Slice Z={slice_idx}')
axes[1].grid(False)

plt.tight_layout()
plt.show()

In [ ]:
slice_idx = original_vol.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(20, 20))

# Plot the original slice z
im1 = axes[0].imshow(original_vol[slice_idx, :256, :256].T, cmap='gray', origin='lower')
axes[0].set_title(f'Original Slice Z={slice_idx}')
axes[0].grid(False)

# Plot the original slice z+1
im2 = axes[1].imshow(denoised_vol[slice_idx, :256, :256].T, cmap='gray', origin='lower')
axes[1].set_title(f'N2N-Odd-Even-Registered Denoised Slice Z={slice_idx}')
axes[1].grid(False)

plt.tight_layout()
plt.show()

In [ ]:
slice_idx = original_vol.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(20, 20))

# Plot the original slice z
im1 = axes[0].imshow(original_vol[slice_idx, 500:1000, 1000:1500].T, cmap='gray', origin='lower')
axes[0].set_title(f'Original Slice Z={slice_idx}')
axes[0].grid(False)

# Plot the original slice z+1
im2 = axes[1].imshow(denoised_vol[slice_idx, 500:1000, 1000:1500].T, cmap='gray', origin='lower')
axes[1].set_title(f'N2N-Odd-Even-Registered Denoised Slice Z={slice_idx}')
axes[1].grid(False)

plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.pyplot import figure
figure(figsize=(16, 16))
slice_idx = denoised_vol.shape[0]//2
plt.imshow(denoised_vol[slice_idx, 200:600, 200:600], cmap="gray")

In [ ]:
figure(figsize=(16, 16))
slice_idx = original_vol.shape[0]//2
plt.imshow(original_vol[slice_idx, 200:600, 200:600], cmap="gray")

In [ ]:
from matplotlib.pyplot import figure
figure(figsize=(16, 16))
slice_idx = denoised_vol.shape[0]//2
plt.imshow(denoised_vol[slice_idx, 200:600, 200:600], cmap="gray")
plt.savefig("denoised.pdf", bbox_inches='tight')